In [ ]:
# =====================================================================
# ASTRO CODE BLUE — DUAL ENDPOINT GPU AI SERVER (/analyze & /chat)
# 1. Install dependencies
# =====================================================================
!pip install -q fastapi uvicorn pydantic transformers torch accelerate nest_asyncio

import json
import torch
import uvicorn
import threading
import subprocess
import time
import re
import os
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# =====================================================================
# 2. Load Qwen 2.5 7B Instruct model onto GPU
# =====================================================================
model_name = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {model_name} onto GPU...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully!")

# =====================================================================
# 3. Define FastAPI App with dual endpoints: /analyze and /chat
# =====================================================================
app = FastAPI(title="ASTRO CODE BLUE AI ENGINE")

class AssessmentPayload(BaseModel):
    raw_assessment: dict

class ChatPayload(BaseModel):
    message: str
    history: list = []
    medical_telemetry: str = ""

# --- Endpoint A: /analyze (Health Assessment Form Submission) ---
@app.post("/analyze")
def analyze(payload: AssessmentPayload):
    data_str = json.dumps(payload.raw_assessment, default=str)
    
    prompt = f"""You are an expert aerospace medicine flight surgeon assistant for the ASTRO CODE BLUE astronaut health system.
Analyze the following astronaut assessment raw telemetry data:
{data_str}

Evaluate overall crew health, assess each section, check for spaceflight hazards, determine medical severity, and recommend 5 exercise countermeasures.

Return ONLY a valid JSON object with EXACTLY this structure (no markdown, no extra commentary):
{{
  "overall_health_status": "Nominal" | "Caution" | "Critical",
  "overall_summary": "Short 2-3 sentence clinical summary",
  "medical_alert": {{
    "severity": "low" | "moderate" | "high" | "critical",
    "seek_medical_support": true | false,
    "advice": "Clear medical recommendation on whether to consult flight surgeon"
  }},
  "section_statuses": {{
    "sleepAndWellness": {{ "status": "Nominal", "findings": "Summary" }},
    "vitalSigns": {{ "status": "Nominal", "findings": "Summary" }},
    "cognitiveHealth": {{ "status": "Nominal", "findings": "Summary" }},
    "visionAndSymptoms": {{ "status": "Nominal", "findings": "Summary" }},
    "spaceEnvironment": {{ "status": "Nominal", "findings": "Summary" }},
    "specializedMedical": {{ "status": "Nominal", "findings": "Summary" }}
  }},
  "exercises": [
    {{ "name": "ARED Heavy Resistance Protocol", "duration": "35 mins", "purpose": "Axial bone loading", "category": "Axial Resistance", "intensity": "High Resistance", "target": "Spine & femur", "protocol": "4x8 reps" }},
    {{ "name": "T2 Treadmill Aerobic Interval", "duration": "30 mins", "purpose": "Cardiorespiratory", "category": "Cardiorespiratory", "intensity": "HIIT", "target": "VO2 max", "protocol": "4x4 intervals" }},
    {{ "name": "CEVIS Cycling Redistribution", "duration": "20 mins", "purpose": "Venous tone", "category": "Vascular", "intensity": "Moderate", "target": "Lower limbs", "protocol": "120-150W" }},
    {{ "name": "Chibis LBNP & SANS Decompression", "duration": "20 mins", "purpose": "Optic nerve decompression", "category": "SANS", "intensity": "Passive", "target": "Optic sheath", "protocol": "-30 mmHg" }},
    {{ "name": "Neuro-Vestibular Balance", "duration": "15 mins", "purpose": "Sensorimotor recalibration", "category": "Neuro-Vestibular", "intensity": "Low Strain", "target": "Vestibular reflex", "protocol": "Tracking drills" }}
  ],
  "healthcare_suggestions": [
    "Maintain 2.8L hydration.",
    "Dim cabin lighting 45 minutes prior to sleep."
  ]
}}
"""
    messages = [
        {"role": "system", "content": "You are a clinical aerospace medicine AI. Always reply with valid JSON only."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**model_inputs, max_new_tokens=768, temperature=0.2, do_sample=False)
    
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    clean_text = response_text.strip()
    if clean_text.startswith("```json"): clean_text = clean_text[7:]
    if clean_text.startswith("```"): clean_text = clean_text[3:]
    if clean_text.endswith("```"): clean_text = clean_text[:-3]
    try:
        return json.loads(clean_text.strip())
    except Exception:
        return {"overall_health_status": "Caution", "overall_summary": clean_text[:200], "medical_alert": {"severity": "moderate", "seek_medical_support": False, "advice": "Telemetry logged."}, "exercises": [], "healthcare_suggestions": []}

# --- Endpoint B: /chat (Fast Tele-Triage with Live DB Telemetry & Memory) ---
@app.post("/chat")
def chat(payload: ChatPayload):
    system_prompt = (
        "You are the ASTRO CODE BLUE Clinical Medical Support Assistant aboard an active space mission.\n"
        "You evaluate astronaut inquiries, symptoms, vitals, and microgravity adaptation.\n"
    )
    if payload.medical_telemetry:
        system_prompt += (
            "\n=== ASTRONAUT LIVE MEDICAL DATABASE TELEMETRY (astro_code_blue.db) ===\n"
            f"{payload.medical_telemetry}\n"
            "=======================================================================\n"
            "Refer to the astronaut's actual medical values above when answering questions.\n"
        )
    system_prompt += "\nProvide concise, direct flight surgeon guidance in 2-4 focused paragraphs."

    messages = [{"role": "system", "content": system_prompt}]
    if isinstance(payload.history, list):
        for m in payload.history[-6:]:
            if isinstance(m, dict) and m.get("content"):
                role = "assistant" if m.get("role") in ["assistant", "ai", "model"] else "user"
                messages.append({"role": role, "content": str(m["content"])})
    messages.append({"role": "user", "content": payload.message})

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**model_inputs, max_new_tokens=350, temperature=0.3, do_sample=True, top_p=0.9)

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    reply = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return {"reply": reply.strip()}

# =====================================================================
# 4. Clean up lingering processes and stale tunnel logs
# =====================================================================
!pkill -9 -f cloudflared 2>/dev/null || true
!pkill -9 -f uvicorn 2>/dev/null || true
!fuser -k 8000/tcp 2>/dev/null || true
if os.path.exists("tunnel.log"):
    os.remove("tunnel.log")
time.sleep(1)

# =====================================================================
# 5. Start Uvicorn Server in background thread (clean port 8000)
# =====================================================================
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(1)

# =====================================================================
# 6. Expose FastAPI using Cloudflare Tunnel (Free, No Account Needed)
# =====================================================================
if not os.path.exists("./cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--logfile", "tunnel.log"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# =====================================================================
# 7. Retrieve and display public URL
# =====================================================================
print("Waiting for Cloudflare Tunnel to initialize...")
public_url = None
for _ in range(25):
    time.sleep(2)
    try:
        with open("tunnel.log", "r") as f:
            content = f.read()
            urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if urls:
                public_url = urls[-1]  # Always pick the newest URL
                break
    except Exception:
        pass

if public_url:
    print("\n" + "="*60)
    print(">>> YOUR PUBLIC KAGGLE API URL IS:")
    print(public_url)
    print("="*60 + "\n")
else:
    print("Tunnel initialized. Check tunnel.log for URL.")

print("Server is active and listening on /analyze and /chat.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Server stopped.")